# Filmory: reproducible training and honest evaluation

This replaces the original notebook's model contracts and training loop. Use a GPU runtime for larger budgets. Low training loss alone does not establish recommendation accuracy. Checkpoints are selected by validation NDCG; sampled and full-catalogue test results remain separate. This notebook embeds the source and runs independently of the Filmory database and API keys.

Run all cells in order. Download the official MovieLens 20M archive from https://grouplens.org/datasets/movielens/20m/ and extract `ratings.csv` and `movies.csv` into the data directory below. Dataset access and terms remain those of GroupLens. No results are fabricated or prefilled.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'numpy', 'pandas', 'matplotlib'])


In [ ]:
from pathlib import Path
import json, datetime
workspace = Path.cwd() / "filmory_training"
files = {'app/__init__.py': '', 'app/ml/__init__.py': '', 'app/ml/architectures.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass NCFBaseline(nn.Module):\n    def __init__(self, num_users: int = 41547, num_items: int = 22836, embedding_dim: int = 8):\n        super().__init__()\n        self.user_embedding = nn.Embedding(num_users, embedding_dim)\n        self.item_embedding = nn.Embedding(num_items, embedding_dim)\n        self.fc1 = nn.Linear(embedding_dim * 2, 64)\n        self.fc2 = nn.Linear(64, 32)\n        self.output = nn.Linear(32, 1)\n\n    def logits(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:\n        u_emb = self.user_embedding(user_idx)\n        i_emb = self.item_embedding(item_idx)\n        x = torch.cat([u_emb, i_emb], dim=-1)\n        x = F.relu(self.fc1(x))\n        x = F.relu(self.fc2(x))\n        return self.output(x).squeeze(-1)\n\n    def forward(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.logits(user_idx, item_idx))\n\n    def score_all_items_for_user(self, user_idx_int: int, device: torch.device) -> torch.Tensor:\n        """Efficient batch scoring of all items for a single user"""\n        num_items = self.item_embedding.num_embeddings\n        user_tensor = torch.full((num_items,), user_idx_int, dtype=torch.long, device=device)\n        item_tensor = torch.arange(num_items, dtype=torch.long, device=device)\n        return self.forward(user_tensor, item_tensor)\n\n\nclass NCFHybrid(nn.Module):\n    def __init__(\n        self,\n        num_users: int = 41547,\n        num_items: int = 22836,\n        num_genres: int = 20,\n        embedding_dim: int = 8,\n    ):\n        super().__init__()\n        self.user_embedding = nn.Embedding(num_users, embedding_dim)\n        self.item_embedding = nn.Embedding(num_items, embedding_dim)\n        self.user_genre_projection = nn.Linear(num_genres, embedding_dim)\n        self.item_genre_projection = nn.Linear(num_genres, embedding_dim)\n        self.fc1 = nn.Linear(embedding_dim * 4, 64)\n        self.fc2 = nn.Linear(64, 32)\n        self.output = nn.Linear(32, 1)\n\n    def logits(\n        self,\n        user_idx: torch.Tensor,\n        item_idx: torch.Tensor,\n        user_genre: torch.Tensor,\n        item_genre: torch.Tensor,\n    ) -> torch.Tensor:\n        u_emb = self.user_embedding(user_idx)\n        i_emb = self.item_embedding(item_idx)\n        # Notebook training used a genre probability distribution (L1 norm).\n        # Serving also uses L2 profiles for cosine similarity; restore the\n        # training scale before feeding the learned genre projection.\n        user_genre = F.normalize(user_genre, p=1, dim=-1)\n        ug_proj = self.user_genre_projection(user_genre)\n        ig_proj = self.item_genre_projection(item_genre)\n        # This order is part of the checkpoint contract (notebook cell 71).\n        x = torch.cat([u_emb, ug_proj, i_emb, ig_proj], dim=-1)\n        x = F.relu(self.fc1(x))\n        x = F.relu(self.fc2(x))\n        return self.output(x).squeeze(-1)\n\n    def forward(self, user_idx, item_idx, user_genre, item_genre) -> torch.Tensor:\n        return torch.sigmoid(self.logits(user_idx, item_idx, user_genre, item_genre))\n\n    def score_candidate_items(\n        self,\n        user_idx_int: int,\n        candidate_item_indices: torch.Tensor,\n        user_genre_vector: torch.Tensor,\n        movie_genre_matrix: torch.Tensor,\n        device: torch.device,\n    ) -> torch.Tensor:\n        """Batch score specific candidate items for a user"""\n        k = len(candidate_item_indices)\n        user_tensor = torch.full((k,), user_idx_int, dtype=torch.long, device=device)\n        user_genre_expanded = user_genre_vector.unsqueeze(0).expand(k, -1).to(device)\n        item_genre_batch = movie_genre_matrix[candidate_item_indices].to(device)\n        \n        return self.forward(\n            user_tensor,\n            candidate_item_indices.to(device),\n            user_genre_expanded,\n            item_genre_batch,\n        )\n\n\nclass SequentialTransformer(nn.Module):\n    def __init__(\n        self,\n        num_items: int = 22836,\n        embedding_dim: int = 64,\n        num_heads: int = 4,\n        num_layers: int = 2,\n        max_len: int = 20,\n        pooling: str = "legacy_length",\n        dim_feedforward: int = 2048,\n    ):\n        super().__init__()\n        self.num_items = num_items\n        self.max_len = max_len\n        self.padding_idx = num_items\n        if pooling not in {"legacy_length", "last"}:\n            raise ValueError("Unknown sequence pooling convention")\n        self.pooling = pooling\n        # Movie indices are zero-based. The final row, not row zero, is padding.\n        self.item_embedding = nn.Embedding(num_items + 1, embedding_dim, padding_idx=self.padding_idx)\n        self.position_embedding = nn.Embedding(max_len, embedding_dim)\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=embedding_dim,\n            nhead=num_heads,\n            dim_feedforward=dim_feedforward,\n            batch_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)\n        self.layer_norm = nn.LayerNorm(embedding_dim)\n\n    def forward(self, sequence_tensor: torch.Tensor) -> torch.Tensor:\n        # sequence_tensor: (batch_size, seq_len)\n        batch_size, seq_len = sequence_tensor.shape\n        positions = torch.arange(seq_len, device=sequence_tensor.device).unsqueeze(0).expand(batch_size, -1)\n        x = self.item_embedding(sequence_tensor) + self.position_embedding(positions)\n        mask = sequence_tensor == self.padding_idx\n        empty = mask.all(dim=1)\n        # All-padding attention is undefined. Unmask one position, then zero\n        # its representation; callers receive a neutral score for no history.\n        safe_mask = mask.clone()\n        safe_mask[empty, -1] = False\n        out = self.transformer(x, src_key_padding_mask=safe_mask)\n        out = self.layer_norm(out)\n        if self.pooling == "legacy_length":\n            # Reproduce the original checkpoint\'s training behavior exactly.\n            # Its left-padding/length-minus-one mismatch is fixed for NEW runs\n            # by selecting pooling="last", rather than silently changing weights.\n            positions = (~mask).sum(1).clamp(min=1) - 1\n        else:\n            positions = torch.arange(seq_len, device=sequence_tensor.device).expand(batch_size, -1)\n            positions = positions.masked_fill(mask, -1).max(1).values.clamp(min=0)\n        result = out[torch.arange(batch_size, device=sequence_tensor.device), positions]\n        return result.masked_fill(empty.unsqueeze(1), 0.0)\n\n    def score_items(self, sequences: torch.Tensor, candidate_items: torch.Tensor) -> torch.Tensor:\n        if sequences.dim() == 1:\n            sequences = sequences.unsqueeze(0)\n        if candidate_items.dim() == 1:\n            candidate_items = candidate_items.unsqueeze(0)\n        representation = self.forward(sequences)\n        return (representation.unsqueeze(1) * self.item_embedding(candidate_items)).sum(-1)\n\n    def score_candidates_with_sequence(\n        self,\n        sequence_item_indices: list[int],\n        candidate_item_indices: torch.Tensor,\n        device: torch.device,\n    ) -> torch.Tensor:\n        """\n        Calculates similarity between the user sequence representation and candidate items.\n        Both history and candidates use zero-based model item indices.\n        """\n        # Trim / pad sequence to max_len\n        seq = sequence_item_indices[-self.max_len :] if sequence_item_indices else []\n        if len(seq) < self.max_len:\n            seq = [self.padding_idx] * (self.max_len - len(seq)) + seq\n        \n        seq_tensor = torch.tensor([seq], dtype=torch.long, device=device)\n        logits = self.score_items(seq_tensor, candidate_item_indices.to(device)).squeeze(0)\n        return torch.sigmoid(logits)\n', 'scripts/train_models.py': '"""Fresh, reproducible training from ordered zero-based histories or MovieLens CSV.\n\nNever fine-tunes shipped checkpoints. Outputs an isolated bundle and real logs.\nRun `python scripts/train_models.py --help` from backend for budget controls.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport pickle\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nBACKEND = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(BACKEND))\nfrom app.ml.architectures import NCFBaseline, NCFHybrid, SequentialTransformer\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef load_data(source, ratings_csv=None):\n    mappings = {}\n    for name in ["user2idx", "idx2user", "movie2idx", "idx2movie", "genre2idx", "idx2genre"]:\n        with (source / f"{name}.pkl").open("rb") as f:\n            mappings[name] = pickle.load(f)\n    with (source / "user_sequences.pkl").open("rb") as f:\n        sequences = pickle.load(f)\n    lineage = {"kind": "ordered stored training histories", "source_sha256": sha(source / "user_sequences.pkl"),\n               "timestamps_available": False,\n               "note": "Original notebook test ratings are absent from these histories. New holdouts are taken before fresh training; chronology is ordinal only."}\n    if (source / "source_provenance.json").exists():\n        lineage = json.loads((source / "source_provenance.json").read_text(encoding="utf-8"))\n    if ratings_csv:\n        import pandas as pd\n        frame = pd.read_csv(ratings_csv, usecols=["userId", "movieId", "timestamp"])\n        frame = frame[frame.userId.isin(mappings["user2idx"]) & frame.movieId.isin(mappings["movie2idx"])]\n        frame = frame.sort_values(["userId", "timestamp", "movieId"], kind="stable")\n        sequences = {mappings["user2idx"][int(u)]: [mappings["movie2idx"][int(i)] for i in g.movieId]\n                     for u, g in frame.groupby("userId", sort=True)}\n        lineage = {"kind": "MovieLens ratings CSV with existing selected-user/item maps", "source_sha256": sha(ratings_csv),\n                   "timestamps_available": True, "note": "Timestamp ordering; tied timestamps broken by movie ID."}\n    genres = torch.load(source / "movie_genre_matrix.pt", map_location="cpu", weights_only=True).float()\n    return mappings, sequences, genres, lineage\n\n\ndef split_sequences(sequences, num_items):\n    train, val, test = {}, {}, {}\n    for u, seq in sorted(sequences.items()):\n        seq = [int(i) for i in seq]\n        if len(seq) < 5:\n            continue\n        if not all(0 <= i < num_items for i in seq):\n            raise ValueError("Sequence contains a canonical ID instead of a model index")\n        if len(set(seq)) != len(seq):\n            raise ValueError("Repeated ratings need explicit deduplication before leave-two-out")\n        train[int(u)], val[int(u)], test[int(u)] = seq[:-2], seq[-2], seq[-1]\n    return train, val, test\n\n\ndef pair_keys(histories, num_items):\n    return np.sort(np.concatenate([np.asarray(s, dtype=np.int64) + int(u)*num_items for u, s in histories.items()]))\n\n\ndef negatives(users, count, num_items, excluded, rng):\n    users = np.asarray(users, dtype=np.int64)\n    shape = (len(users), count)\n    out = rng.integers(num_items, size=shape, dtype=np.int64)\n    for _ in range(100):\n        keys = users[:, None]*num_items + out\n        pos = np.searchsorted(excluded, keys)\n        bad = (pos < len(excluded)) & (excluded[np.minimum(pos, len(excluded)-1)] == keys)\n        if not bad.any():\n            return out\n        out[bad] = rng.integers(num_items, size=int(bad.sum()))\n    raise RuntimeError("Negative rejection sampler exhausted its retry budget")\n\n\ndef padded(histories, padding_idx, max_len):\n    result = np.full((len(histories), max_len), padding_idx, dtype=np.int64)\n    for row, hist in enumerate(histories):\n        h = hist[-max_len:]\n        if h:\n            result[row, -len(h):] = h\n    return result\n\n\ndef rank_metrics(scores, top_k=10):\n    # Candidates have target in column zero. Conservative ties: negatives tied\n    # with target precede it. No arbitrary optimistic tie credit.\n    ranks = 1 + (scores[:, 1:] >= scores[:, :1]).sum(1)\n    hit = ranks <= top_k\n    return {"users": int(len(ranks)), "hr10": float(hit.mean()),\n            "ndcg10": float(np.where(hit, 1/np.log2(ranks+1), 0).mean()),\n            "mrr": float((1/ranks).mean())}\n\n\n@torch.inference_mode()\ndef evaluate(model, kind, users, targets, histories, genres, profiles, candidates, device, batch=128):\n    model.eval()\n    results = []\n    for start in range(0, len(users), batch):\n        us = users[start:start+batch]\n        cand = torch.as_tensor(candidates[start:start+batch], device=device)\n        u = torch.as_tensor(us, device=device)[:, None].expand_as(cand)\n        if kind == "transformer":\n            seq = torch.as_tensor(padded([histories[int(x)] for x in us], model.padding_idx, model.max_len), device=device)\n            scores = model.score_items(seq, cand)\n        elif kind == "hybrid":\n            scores = model.logits(u, cand, profiles[u], genres[cand])\n        else:\n            scores = model.logits(u, cand)\n        if not torch.isfinite(scores).all():\n            raise RuntimeError("Non-finite validation predictions")\n        results.append(scores.cpu().numpy())\n    return rank_metrics(np.concatenate(results))\n\n\n@torch.inference_mode()\ndef catalogue_evaluate(models, users, targets, histories, genres, profiles, popularity, device):\n    num_items = len(genres)\n    items = torch.arange(num_items, device=device)\n    per_user = []\n    for user in users:\n        uid = int(user)\n        row = {"user_idx": uid, "target_idx": targets[uid], "systems": {}}\n        raw = {"popularity": torch.as_tensor(popularity, device=device).float()}\n        for kind, model in models.items():\n            model.eval()\n            u = torch.full((num_items,), uid, device=device, dtype=torch.long)\n            if kind == "hybrid":\n                s = model(u, items, profiles[uid].expand(num_items, -1), genres)\n            elif kind == "transformer":\n                s = model.score_candidates_with_sequence(histories[uid], items, device)\n            else:\n                s = model(u, items)\n            raw[kind] = s\n        for kind, scores in raw.items():\n            scores = scores.clone()\n            scores[histories[uid]] = -torch.inf\n            target_score = scores[targets[uid]]\n            rank = int((scores >= target_score).sum())\n            row["systems"][kind] = {"rank": rank, "hr10": int(rank <= 10), "recall100": int(rank <= 100),\n                                    "ndcg10": 1/math.log2(rank+1) if rank <= 10 else 0,\n                                    "top10": scores.topk(10).indices.cpu().tolist()}\n        per_user.append(row)\n    aggregate = {name: {"users": len(users), **{metric: float(np.mean([r["systems"][name][metric] for r in per_user]))\n                                              for metric in ["hr10", "recall100", "ndcg10"]}}\n                 for name in raw}\n    return aggregate, per_user\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--source", type=Path, default=BACKEND / "ml")\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--ratings-csv", type=Path)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--epochs", type=int, default=8)\n    parser.add_argument("--sequence-epochs", type=int, default=5)\n    parser.add_argument("--positives-per-epoch", type=int, default=1000000)\n    parser.add_argument("--sequence-samples", type=int, default=100000)\n    parser.add_argument("--batch-size", type=int, default=4096)\n    parser.add_argument("--eval-users", type=int, default=400)\n    parser.add_argument("--catalogue-users", type=int, default=200)\n    parser.add_argument("--threads", type=int, default=4)\n    parser.add_argument("--embedding-dim", type=int, default=8)\n    parser.add_argument("--test-user-offset", type=int)\n    parser.add_argument("--continue-from", type=Path, help="Warm start from a clean bundle with the same pre-training split; never use shipped legacy checkpoints")\n    parser.add_argument("--learning-rate", type=float, default=0.001)\n    args = parser.parse_args(argv)\n    args.output = args.output.resolve()\n    args.source = args.source.resolve()\n    if args.output == args.source or args.source in args.output.parents and args.output.name == "ml":\n        parser.error("Training output must be an isolated directory")\n    args.output.mkdir(parents=True, exist_ok=False)\n    shutil.copy2(__file__, args.output / "training_source_snapshot.py")\n    shutil.copy2(BACKEND / "app/ml/architectures.py", args.output / "architecture_source_snapshot.py")\n    torch.set_num_threads(args.threads)\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    rng = np.random.default_rng(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    mappings, original, genre_cpu, lineage = load_data(args.source, args.ratings_csv)\n    num_users, num_items = len(mappings["user2idx"]), len(genre_cpu)\n    histories, val, test = split_sequences(original, num_items)\n    parent = args.continue_from.resolve() if args.continue_from else None\n    if parent:\n        parent_split = json.loads((parent / "split.json").read_text(encoding="utf-8"))\n        parent_config = json.loads((parent / "model_config.json").read_text(encoding="utf-8"))\n        if parent_config.get("model_format_version") != 2:\n            raise ValueError("Continuation requires a clean version-2 bundle")\n        if {str(k): v for k,v in val.items()} != parent_split["validation_targets"] or {str(k): v for k,v in test.items()} != parent_split["test_targets"]:\n            raise ValueError("Continuation targets differ from parent; checkpoints could contain holdout information")\n        with (parent / "user_sequences.pkl").open("rb") as f:\n            if histories != pickle.load(f):\n                raise ValueError("Continuation training histories differ from parent")\n        if args.embedding_dim != parent_config["embedding_dim_ncf"]:\n            raise ValueError("Continuation embedding dimension must match the parent")\n        for name in ["user2idx", "movie2idx", "genre2idx"]:\n            with (parent / f"{name}.pkl").open("rb") as f:\n                if mappings[name] != pickle.load(f): raise ValueError("Continuation mappings differ from parent")\n    users_all = np.asarray(sorted(histories), dtype=np.int64)\n    if len(users_all) < 2:\n        raise ValueError("At least two eligible users are required")\n    order = rng.permutation(users_all)\n    n_eval = min(args.eval_users, len(order)//2)\n    test_offset = args.test_user_offset if args.test_user_offset is not None else n_eval\n    if test_offset < n_eval or test_offset+n_eval > len(order):\n        parser.error("Test-user offset must leave disjoint, complete evaluation sets")\n    val_users, test_users = order[:n_eval], order[test_offset:test_offset+n_eval]\n    train_keys = pair_keys(histories, num_items)\n    # Training negatives exclude TRAIN ONLY; held-out labels cannot affect fitting.\n    # Evaluation negatives exclude all known positives and are sampled without replacement.\n    def eval_candidates(us, targets, seed):\n        erng = np.random.default_rng(seed)\n        rows = []\n        for u in us:\n            banned = set(histories[int(u)]) | {val[int(u)], test[int(u)]}\n            available = np.asarray([i for i in range(num_items) if i not in banned])\n            neg = erng.choice(available, size=min(99, len(available)), replace=False)\n            rows.append(np.concatenate([[targets[int(u)]], neg]))\n        if len({len(x) for x in rows}) != 1:\n            raise ValueError("Insufficient unseen evaluation candidates")\n        return np.asarray(rows)\n    val_candidates = eval_candidates(val_users, val, args.seed+101)\n    test_candidates = eval_candidates(test_users, test, args.seed+202)\n    np.savez_compressed(args.output / "evaluation_candidates.npz", val_users=val_users, test_users=test_users,\n                        val_candidates=val_candidates, test_candidates=test_candidates)\n    profile_cpu = torch.zeros(num_users, genre_cpu.shape[1])\n    popularity = np.zeros(num_items, dtype=np.int64)\n    for uid, hist in histories.items():\n        profile_cpu[uid] = genre_cpu[hist].sum(0)\n        np.add.at(popularity, hist, 1)\n    profile_cpu = F.normalize(profile_cpu, p=1, dim=1)\n    genres, profiles = genre_cpu.to(device), profile_cpu.to(device)\n    np.save(args.output / "train_popularity.npy", popularity)\n    for name, mapping in mappings.items():\n        with (args.output / f"{name}.pkl").open("wb") as f:\n            pickle.dump(mapping, f)\n    for name in ["user_sequences", "user_interacted"]:\n        with (args.output / f"{name}.pkl").open("wb") as f:\n            pickle.dump(histories, f)\n    torch.save(genre_cpu, args.output / "movie_genre_matrix.pt")\n    torch.save(profile_cpu, args.output / "user_genre_matrix.pt")\n    shutil.copy2(args.source / "movies_metadata.csv", args.output / "movies_metadata.csv")\n    (args.output / "split.json").write_text(json.dumps({"seed": args.seed, "validation_targets": val, "test_targets": test,\n                                                       "validation_users": val_users.tolist(), "test_users": test_users.tolist()}), encoding="utf-8")\n    config = json.loads((args.source / "model_config.json").read_text(encoding="utf-8"))\n    config.update(transformer_pooling="last", transformer_dim_feedforward=256, transformer_padding_idx=num_items,\n                  model_format_version=2, history_indexing="zero_based", genre_profile_normalization="l1",\n                  embedding_dim_ncf=args.embedding_dim)\n    (args.output / "model_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")\n    provenance = {"fresh_initialization": True, "source": lineage, "seed": args.seed, "device": str(device),\n                  "torch_version": torch.__version__, "eligible_users": len(histories),\n                  "training_interactions": sum(map(len, histories.values())),\n                  "training_negative_exclusion": "train histories only",\n                  "selection": "sampled validation NDCG@10; test not used for selection",\n                  "budget": {k: v for k, v in vars(args).items() if isinstance(v, (int, float, str))},\n                  "architecture_sha256": sha(BACKEND / "app/ml/architectures.py"), "training_script_sha256": sha(__file__)}\n    if parent:\n        provenance.update(fresh_initialization=False, continuation_from=str(parent),\n                          parent_checkpoint_sha256={name:sha(parent/name) for name in ["ncf_baseline.pth", "ncf_hybrid.pth", "sequential_transformer.pth"]},\n                          note="Fresh optimizer; continued fitting on identical clean training histories. Original parent checkpoint retained unless validation NDCG improves.")\n    (args.output / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")\n    print(json.dumps(provenance), flush=True)\n    positive_users = np.repeat(users_all, [len(histories[int(u)]) for u in users_all])\n    positive_items = np.concatenate([np.asarray(histories[int(u)]) for u in users_all])\n    models = {}\n    histories_log = {}\n    selected_checkpoints = {}\n    for kind in ["baseline", "hybrid", "transformer"]:\n        torch.manual_seed(args.seed)\n        if kind == "baseline":\n            model = NCFBaseline(num_users, num_items, embedding_dim=config.get("embedding_dim_ncf", 8))\n        elif kind == "hybrid":\n            model = NCFHybrid(num_users, num_items, len(genres[0]), config.get("embedding_dim_ncf", 8))\n            # Warm start from this run\'s CLEAN baseline, not the shipped checkpoint.\n            base = models["baseline"]\n            model.user_embedding.weight.data.copy_(base.user_embedding.weight.detach().cpu())\n            model.item_embedding.weight.data.copy_(base.item_embedding.weight.detach().cpu())\n            model.fc1.weight.data.zero_()\n            dim = model.user_embedding.embedding_dim\n            model.fc1.weight.data[:, :dim].copy_(base.fc1.weight.data[:, :dim].cpu())\n            model.fc1.weight.data[:, 2*dim:3*dim].copy_(base.fc1.weight.data[:, dim:].cpu())\n            for layer in ["fc1", "fc2", "output"]:\n                getattr(model, layer).bias.data.copy_(getattr(base, layer).bias.data.cpu())\n            model.fc2.weight.data.copy_(base.fc2.weight.data.cpu())\n            model.output.weight.data.copy_(base.output.weight.data.cpu())\n        else:\n            model = SequentialTransformer(num_items, config.get("transformer_embedding_dim",64),\n                                          config.get("transformer_heads",4), config.get("transformer_layers",2),\n                                          config.get("transformer_max_len",20), pooling="last", dim_feedforward=256)\n            torch.nn.init.normal_(model.item_embedding.weight, std=0.02)\n            torch.nn.init.normal_(model.position_embedding.weight, std=0.02)\n            model.item_embedding.weight.data[model.padding_idx].zero_()\n        model = model.to(device)\n        filename = "sequential_transformer.pth" if kind == "transformer" else f"ncf_{kind}.pth"\n        if parent:\n            model.load_state_dict(torch.load(parent/filename, map_location=device, weights_only=True))\n        opt = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=1e-5)\n        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=1, factor=0.5)\n        best, stale = -1.0, 0\n        if parent:\n            initial = evaluate(model,kind,val_users,val,histories,genres,profiles,val_candidates,device)\n            best = initial["ndcg10"]\n            torch.save(model.state_dict(), args.output/filename)\n            selected_checkpoints[kind] = {"origin":"parent", "validation":initial}\n            print(json.dumps({"model":kind,"initial_validation":initial}), flush=True)\n        log = []\n        filename = "sequential_transformer.pth" if kind == "transformer" else f"ncf_{kind}.pth"\n        epochs = args.sequence_epochs if kind == "transformer" else args.epochs\n        for epoch in range(1, epochs+1):\n            began = time.perf_counter()\n            model.train()\n            if kind == "transformer":\n                # Uniform users rather than the notebook\'s first-300k interaction\n                # cutoff, which omitted most users from sequence training.\n                sample_users = rng.choice(users_all, size=args.sequence_samples, replace=True)\n                positions = np.asarray([rng.integers(1, len(histories[int(u)])) for u in sample_users])\n                xs = padded([histories[int(u)][max(0,p-model.max_len):p] for u,p in zip(sample_users,positions)], num_items, model.max_len)\n                ys = np.asarray([histories[int(u)][p] for u,p in zip(sample_users,positions)])\n                batch_size = 256\n            else:\n                chosen = rng.choice(len(positive_items), size=min(args.positives_per_epoch, len(positive_items)), replace=False)\n                sample_users, ys = positive_users[chosen], positive_items[chosen]\n                batch_size = args.batch_size\n            loss_sum, bce_sum, total = 0.0, 0.0, 0\n            for start in range(0, len(ys), batch_size):\n                us = sample_users[start:start+batch_size]\n                cand_np = np.concatenate([ys[start:start+batch_size,None], negatives(us,4,num_items,train_keys,rng)],axis=1)\n                cand = torch.as_tensor(cand_np, device=device)\n                u = torch.as_tensor(us, device=device)[:,None].expand_as(cand)\n                opt.zero_grad(set_to_none=True)\n                if kind == "transformer":\n                    scores = model.score_items(torch.as_tensor(xs[start:start+batch_size],device=device),cand)\n                elif kind == "hybrid":\n                    scores = model.logits(u,cand,profiles[u],genres[cand])\n                else:\n                    scores = model.logits(u,cand)\n                labels = torch.zeros_like(scores); labels[:,0] = 1\n                bce = F.binary_cross_entropy_with_logits(scores,labels)\n                # Joint classification/ranking objective: do not optimize a\n                # misleading "accuracy percentage" by predicting negatives only.\n                loss = bce + 0.25*F.softplus(scores[:,1:]-scores[:,:1]).mean()\n                if not torch.isfinite(loss): raise RuntimeError("Non-finite training loss")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n                opt.step()\n                count=len(us); total+=count; loss_sum+=float(loss.detach())*count; bce_sum+=float(bce.detach())*count\n            validation = evaluate(model,kind,val_users,val,histories,genres,profiles,val_candidates,device)\n            scheduler.step(validation["ndcg10"])\n            record={"model":kind,"epoch":epoch,"loss":loss_sum/total,"bce":bce_sum/total,\n                    "positives":total,"validation":validation,"seconds":time.perf_counter()-began,\n                    "lr":opt.param_groups[0]["lr"]}\n            log.append(record)\n            with (args.output/"training_log.jsonl").open("a",encoding="utf-8") as f: f.write(json.dumps(record)+"\\n")\n            print(json.dumps(record),flush=True)\n            if validation["ndcg10"] > best:\n                best=validation["ndcg10"]; stale=0\n                torch.save(model.state_dict(),args.output/filename)\n                selected_checkpoints[kind] = {"origin":"this_run", "epoch":epoch, "bce":record["bce"], "loss":record["loss"], "validation":validation}\n            else: stale+=1\n            if stale>=3: break\n        model.load_state_dict(torch.load(args.output/filename,map_location=device,weights_only=True))\n        model.eval(); models[kind]=model; histories_log[kind]=log\n    sampled={kind:evaluate(m,kind,test_users,test,histories,genres,profiles,test_candidates,device) for kind,m in models.items()}\n    # Validation catalogue checks happen before evaluating the test catalogue.\n    cv=val_users[:min(args.catalogue_users,len(val_users))]\n    ct=test_users[:min(args.catalogue_users,len(test_users))]\n    val_full,val_rows=catalogue_evaluate(models,cv,val,histories,genres,profiles,popularity,device)\n    preferred=max(models,key=lambda name:(val_full[name]["ndcg10"],val_full[name]["recall100"]))\n    config["preferred_retriever"]=preferred\n    (args.output/"model_config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")\n    test_full,test_rows=catalogue_evaluate(models,ct,test,histories,genres,profiles,popularity,device)\n    for name,records in [("validation_catalogue.jsonl",val_rows),("test_catalogue.jsonl",test_rows)]:\n        (args.output/name).write_text("".join(json.dumps(r)+"\\n" for r in records),encoding="utf-8")\n    result={"protocol":"fresh leave-two-out training; sampled and full-catalogue metrics reported separately",\n            "sampled_test":sampled,"full_catalogue_validation":val_full,"full_catalogue_test":test_full,\n            "preferred_retriever_from_validation":preferred,"epochs":histories_log,\n            "selected_checkpoints":selected_checkpoints,\n            "limitations":["Single seed; bounded per-epoch sampling budget", "No original notebook test ratings unless CSV supplied",\n                            "No independent human assessment or publication certification"]}\n    (args.output/"metrics.json").write_text(json.dumps(result,indent=2),encoding="utf-8")\n    print(json.dumps({k:v for k,v in result.items() if k!="epochs"}),flush=True)\n    return result\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/prepare_training_data.py': '"""Build a zero-based source bundle from original MovieLens CSV files.\n\nThe source bundle is NOT a trained serving bundle. train_models.py performs\nleave-two-out BEFORE fitting models or user profiles.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport pickle\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\n\n\ndef prepare(ratings_csv, movies_csv, output, fraction=0.30, seed=42):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    ratings = pd.read_csv(ratings_csv, usecols=["userId", "movieId", "timestamp"])\n    movies = pd.read_csv(movies_csv, usecols=["movieId", "title", "genres"]).fillna({"genres":"(no genres listed)"})\n    users = ratings.userId.unique()\n    if not 0 < fraction <= 1: raise ValueError("User fraction must be in (0,1]")\n    selected = np.random.RandomState(seed).choice(users, size=max(1,int(len(users)*fraction)), replace=False)\n    ratings = ratings[ratings.userId.isin(selected)].sort_values(["userId","timestamp","movieId"],kind="stable")\n    if ratings.duplicated(["userId","movieId"]).any():\n        raise ValueError("Duplicate user/movie ratings need explicit preprocessing")\n    user2idx={int(u):i for i,u in enumerate(sorted(ratings.userId.unique()))}\n    movie2idx={int(m):i for i,m in enumerate(sorted(ratings.movieId.unique()))}\n    genres=sorted({g for s in movies.genres for g in s.split("|")})\n    genre2idx={g:i for i,g in enumerate(genres)}\n    mappings={"user2idx":user2idx,"movie2idx":movie2idx,"genre2idx":genre2idx,\n              "idx2user":{i:u for u,i in user2idx.items()},"idx2movie":{i:m for m,i in movie2idx.items()},\n              "idx2genre":{i:g for g,i in genre2idx.items()}}\n    for name,mapping in mappings.items():\n        with (output/f"{name}.pkl").open("wb") as f:pickle.dump(mapping,f)\n    sequences={user2idx[int(u)]:[movie2idx[int(m)] for m in group.movieId] for u,group in ratings.groupby("userId",sort=True)}\n    with (output/"user_sequences.pkl").open("wb") as f:pickle.dump(sequences,f)\n    matrix=torch.zeros(len(movie2idx),len(genres))\n    for row in movies.itertuples(index=False):\n        if int(row.movieId) in movie2idx:\n            for g in row.genres.split("|"):matrix[movie2idx[int(row.movieId)],genre2idx[g]]=1\n    if (matrix.sum(1)==0).any():raise ValueError("Mapped items are missing movie metadata")\n    torch.save(matrix,output/"movie_genre_matrix.pt")\n    movies.to_csv(output/"movies_metadata.csv",index=False)\n    config={"num_users":len(user2idx),"num_items":len(movie2idx),"num_genres":len(genres),\n            "embedding_dim_ncf":16,"transformer_embedding_dim":64,"transformer_heads":4,\n            "transformer_layers":2,"transformer_max_len":20,"candidate_k":100,"top_k":10,\n            "ncf_weight":0.55,"transformer_weight":0.25,"genre_weight":0.20}\n    (output/"model_config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")\n    provenance={"kind":"original MovieLens CSV source bundle before splitting", "timestamps_available":True,\n                "seed":seed,"user_fraction":fraction,"selected_interactions":len(ratings),\n                "source_sha256":{Path(p).name:hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in [ratings_csv,movies_csv]},\n                "note":"Sequences retain timestamp order (ties broken by movieId); split before training. Metadata maps include the selected transductive catalogue."}\n    (output/"source_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")\n    return provenance\n\n\nif __name__=="__main__":\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--ratings-csv",type=Path,required=True);p.add_argument("--movies-csv",type=Path,required=True)\n    p.add_argument("--output",type=Path,required=True);p.add_argument("--user-fraction",type=float,default=.30)\n    p.add_argument("--seed",type=int,default=42)\n    a=p.parse_args();print(json.dumps(prepare(a.ratings_csv,a.movies_csv,a.output,a.user_fraction,a.seed),indent=2))\n', 'scripts/compare_finetuning.py': '"""Compare parent and continuation on IDENTICAL validation data, not test data."""\nimport argparse\nimport json\nimport pickle\nimport sys\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom train_models import evaluate, padded, negatives, pair_keys, sha\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom app.ml.architectures import NCFBaseline, NCFHybrid, SequentialTransformer\n\n@torch.inference_mode()\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--parent\',type=Path,required=True);p.add_argument(\'--child\',type=Path,required=True)\n    a=p.parse_args();torch.set_num_threads(4);device=torch.device(\'cpu\')\n    root=a.child;config=json.loads((root/\'model_config.json\').read_text())\n    with (root/\'user_sequences.pkl\').open(\'rb\') as f:histories=pickle.load(f)\n    split=json.loads((root/\'split.json\').read_text());targets={int(k):v for k,v in split[\'validation_targets\'].items()}\n    candidates=np.load(root/\'evaluation_candidates.npz\');users=candidates[\'val_users\'];items=candidates[\'val_candidates\']\n    genres=torch.load(root/\'movie_genre_matrix.pt\',weights_only=True);profiles=torch.load(root/\'user_genre_matrix.pt\',weights_only=True)\n    rng=np.random.default_rng(903);all_users=np.asarray(sorted(histories));probe_users=rng.choice(all_users,8000)\n    probe_items=np.asarray([rng.choice(histories[int(u)][1:]) for u in probe_users])\n    probe_candidates=np.concatenate([probe_items[:,None],negatives(probe_users,4,len(genres),pair_keys(histories,len(genres)),rng)],axis=1)\n    result={}\n    for label,bundle in [(\'parent\',a.parent),(\'finetuned\',a.child)]:\n        results={}\n        for kind in [\'baseline\',\'hybrid\',\'transformer\']:\n            if kind==\'baseline\':model=NCFBaseline(config[\'num_users\'],len(genres),config[\'embedding_dim_ncf\'])\n            elif kind==\'hybrid\':model=NCFHybrid(config[\'num_users\'],len(genres),len(genres[0]),config[\'embedding_dim_ncf\'])\n            else:model=SequentialTransformer(len(genres),config[\'transformer_embedding_dim\'],config[\'transformer_heads\'],config[\'transformer_layers\'],config[\'transformer_max_len\'],pooling=\'last\',dim_feedforward=config[\'transformer_dim_feedforward\'])\n            filename=\'sequential_transformer.pth\' if kind==\'transformer\' else \'ncf_\'+kind+\'.pth\'\n            model.load_state_dict(torch.load(bundle/filename,weights_only=True));model.eval()\n            results[kind]=evaluate(model,kind,users,targets,histories,genres,profiles,items,device)\n            # Fixed validation BCE with one positive and four negatives; no class-ratio change.\n            validation_losses=[];probe_losses=[]\n            for population,us,cands,destination in [(\'validation\',users,items[:,:5],validation_losses),(\'training\',probe_users,probe_candidates,probe_losses)]:\n                for start in range(0,len(us),128):\n                    uid=torch.as_tensor(us[start:start+128]);cand=torch.as_tensor(cands[start:start+128]);u=uid[:,None].expand_as(cand)\n                    if kind==\'transformer\':\n                        if population==\'training\':\n                            prefix=[histories[int(x)][:histories[int(x)].index(int(y))] for x,y in zip(uid,cand[:,0])]\n                        else:prefix=[histories[int(x)] for x in uid]\n                        scores=model.score_items(torch.as_tensor(padded(prefix,len(genres),model.max_len)),cand)\n                    elif kind==\'hybrid\':scores=model.logits(u,cand,profiles[u],genres[cand])\n                    else:scores=model.logits(u,cand)\n                    truth=torch.zeros_like(scores);truth[:,0]=1\n                    loss=torch.nn.functional.binary_cross_entropy_with_logits(scores,truth,reduction=\'none\').mean(1)\n                    destination.extend(loss.tolist())\n            results[kind][\'validation_bce_fixed4neg\']=float(np.mean(validation_losses))\n            results[kind][\'training_probe_bce_fixed4neg\']=float(np.mean(probe_losses))\n            results[kind][\'checkpoint_sha256\']=sha(bundle/filename)\n        result[label]=results\n    result[\'protocol\']=\'Identical 400-user validation ranking and one-positive/four-negative BCE; identical 8000 training probes in eval mode; no test-based selection. Transformer training probe excludes its target from the input prefix.\'\n    (root/\'finetuning_comparison.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))\nif __name__==\'__main__\':main()\n'}
for relative, source in files.items():
    path = workspace / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
data_dir = Path.cwd() / 'ml-20m'  # change to your extracted CSV directory
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
source_dir = workspace / 'data' / stamp
run_dir = workspace / 'training' / 'runs' / stamp


## Prepare and split before fitting
Select 30% of users with seed 42. Sort ratings by timestamp and break ties by movie ID. The final two interactions become validation and test targets before fitting, and genre profiles are rebuilt from training interactions only. Both targets are scored with the same static training prefix. Mappings use the selected transductive catalogue; this does not evaluate unseen-item cold start.

In [ ]:
assert (data_dir/'ratings.csv').exists() and (data_dir/'movies.csv').exists(), 'Set data_dir to the extracted MovieLens CSV directory'
subprocess.check_call([sys.executable, str(workspace/'scripts/prepare_training_data.py'),
    '--ratings-csv', str(data_dir/'ratings.csv'), '--movies-csv', str(data_dir/'movies.csv'), '--output', str(source_dir)])


## Train the baseline and two learned components
The hybrid starts from the baseline fitted in this clean run. The Transformer samples users uniformly, uses a dedicated padding index, and pools the last valid sequence position. Training uses stable logit BCE plus a pairwise ranking term, gradient clipping, learning-rate reduction, and validation checkpoint selection. Increase budgets only using validation results; reserve untouched test users for the final experiment.

In [ ]:
subprocess.check_call([sys.executable, str(workspace/'scripts/train_models.py'),
    '--source', str(source_dir), '--output', str(run_dir), '--epochs', '10', '--sequence-epochs', '5',
    '--positives-per-epoch', '1500000', '--sequence-samples', '120000', '--embedding-dim', '16',
    '--eval-users', '400', '--catalogue-users', '200', '--threads', '4'])


In [ ]:
metrics = json.loads((run_dir/'metrics.json').read_text())
for name in ['sampled_test', 'full_catalogue_validation', 'full_catalogue_test']:
    print(name, json.dumps(metrics[name], indent=2))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, records in metrics['epochs'].items():
    axes[0].plot([r['epoch'] for r in records], [r['bce'] for r in records], marker='o', label=name)
    axes[1].plot([r['epoch'] for r in records], [r['validation']['ndcg10'] for r in records], marker='o', label=name)
axes[0].set(xlabel='Epoch', ylabel='Training BCE')
axes[1].set(xlabel='Epoch', ylabel='Sampled validation NDCG@10')
for ax in axes: ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(run_dir/'training_curves.png', dpi=160)
plt.show()


## Optional smaller-learning-rate continuation
Set the flag below only when you want an additional experiment. It keeps the clean training histories, preserves the parent whenever validation NDCG does not improve, and uses a new test-user population. The fixed-example comparison reports training-probe and validation BCE under the same one-positive/four-negative ratio. Do not repeatedly tune on the new test results.

In [ ]:
DO_FINE_TUNE = False
if DO_FINE_TUNE:
    fine_dir = run_dir.parent / (stamp + '_fine_tune')
    subprocess.check_call([sys.executable, str(workspace/'scripts/train_models.py'),
        '--source', str(source_dir), '--continue-from', str(run_dir), '--output', str(fine_dir),
        '--embedding-dim', '16', '--epochs', '5', '--sequence-epochs', '5',
        '--learning-rate', '0.0002', '--positives-per-epoch', '1500000', '--sequence-samples', '120000',
        '--test-user-offset', '800', '--eval-users', '400', '--catalogue-users', '200'])
    subprocess.check_call([sys.executable, str(workspace/'scripts/compare_finetuning.py'),
        '--parent', str(run_dir), '--child', str(fine_dir)])


## Reporting and reproducibility
Use `metrics.json`, `training_log.jsonl`, `split.json`, `provenance.json`, candidate arrays, and source snapshots as evidence. Report sample sizes, negative sampling, objective, training budgets, seed and static-prefix protocol. A hit rate against 99 negatives is not overall prediction accuracy. Repeat multiple seeds, increase catalogue evaluation users, assess explanation claims independently, and run a new blinded assistant set before making publication claims. Turnitin and journal acceptance have not been verified by this notebook.